# 9.6 [응용] Text-to-SQL

## 환경 설정 (Colab)

In [ ]:
!pip install langchain_openai==1.1.12 langchain_community==0.4.1 langchain==1.2.14 sqlalchemy numexpr pydantic tenacity nest_asyncio
!pip install -U duckduckgo_search==7.5.1 yfinance ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.4.2
    Uninstalling langgraph-sdk-0.4.2:
    

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"]=userdata.get("OPENAI_API_KEY")
except Exception:
    pass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"]=getpass.getpass("OpenAI API Key: ")

OpenAI API Key: ··········


## 9.6.1 Text-to-SQL 개념

## 9.6.2 [프로젝트] 병원 DB 질의 챗봇

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.agent_toolkits.sql.prompt import SQL_PREFIX
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
import sqlite3
import os

In [ ]:
# 1. 파일 기반 SQLite DB 생성 및 샘플 데이터 주입
db_path = "hospital.db"
# 셀을 여러 번 실행할 때 에러가 나지 않도록 기존 파일 삭제
if os.path.exists(db_path):
    os.remove(db_path)
# ':memory:' 대신 실제 물리적 파일로 DB 생성
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [ ]:
cursor.executescript("""
CREATE TABLE doctors (
    id INTEGER PRIMARY KEY,
    name TEXT,
    specialty TEXT,
    email TEXT
);
CREATE TABLE appointments (
    app_id INTEGER PRIMARY KEY,
    doctor_id INTEGER,
    patient_name TEXT,
    available_time TEXT,
    FOREIGN KEY(doctor_id) REFERENCES doctors(id)
);
-- 의사 3명 삽입
INSERT INTO doctors VALUES (1, '김재준', '내과', 'kim_int@hospital.com');
INSERT INTO doctors VALUES (2, '이수진', '외과', 'lee_surg@hospital.com');
INSERT INTO doctors VALUES (3, '박민수', '소아과', 'park_ped@hospital.com');
-- 예약 5건 삽입 (doctor_id로 의사와 연결)
INSERT INTO appointments VALUES (101, 1, '이나라', '2025-05-10 14:00');
INSERT INTO appointments VALUES (102, 1, '최지훈', '2025-05-11 10:00');
INSERT INTO appointments VALUES (103, 2, '강민준', '2025-05-10 09:00');
INSERT INTO appointments VALUES (104, 3, '오서연', '2025-05-12 15:00');
INSERT INTO appointments VALUES (105, 1, '정하은', '2025-05-13 11:00');
""")
conn.commit()
conn.close()

# 랭체인 표준 SQLDatabase 인터페이스로 연결
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

In [ ]:
# 2. SQL 에이전트에 멀티턴(기억력) 프롬프트 장착
system_message = SQL_PREFIX.format(dialect="SQLite", top_k=5)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    MessagesPlaceholder(variable_name="chat_history"),      # 이전 대화 기억 공간
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [ ]:
# 3. 에이전트 생성 (커스텀 프롬프트 적용)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

agent_executor = create_sql_agent(
    llm=llm,
    db=db,
    prompt=prompt,             # 기억력이 추가된 프롬프트 주입
    agent_type="openai-tools",
    verbose=True
)

In [ ]:
# 4. 메모리 래퍼 적용 (9.4.3절과 동일한 원리)
chat_history = ChatMessageHistory()

conversational_agent = RunnableWithMessageHistory(
    agent_executor,
    lambda session_id: chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
# 5. 실행 및 테스트
print("=== 1차 질문 ===")
response = conversational_agent.invoke(
    {"input": "김재준 의사의 이메일이 뭐야?"},
    config={"configurable": {"session_id": "hospital_session"}}  # 세션 ID로 대화방 구분
)
print(f"답변: {response['output']}\n")

=== 1차 질문 ===


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


appointments, doctors
Invoking: `sql_db_schema` with `{'table_names': 'doctors'}`



CREATE TABLE doctors (
	id INTEGER, 
	name TEXT, 
	specialty TEXT, 
	email TEXT, 
	PRIMARY KEY (id)
)

/*
3 rows from doctors table:
id	name	specialty	email
1	김재준	내과	kim_int@hospital.com
2	이수진	외과	lee_surg@hospital.com
3	박민수	소아과	park_ped@hospital.com
*/김재준 의사의 이메일은 "kim_int@hospital.com"입니다.

> Finished chain.
답변: 김재준 의사의 이메일은 "kim_int@hospital.com"입니다.



In [ ]:
print("=== 2차 질문 ===")
response_2 = conversational_agent.invoke(
    {"input": "그 의사의 예약 가능 시간은 언제야?"},
    config={"configurable": {"session_id": "hospital_session"}}
)
print(f"답변: {response_2['output']}")

=== 2차 질문 ===


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


appointments, doctors
Invoking: `sql_db_schema` with `{'table_names': 'appointments, doctors'}`



CREATE TABLE appointments (
	app_id INTEGER, 
	doctor_id INTEGER, 
	patient_name TEXT, 
	available_time TEXT, 
	PRIMARY KEY (app_id), 
	FOREIGN KEY(doctor_id) REFERENCES doctors (id)
)

/*
3 rows from appointments table:
app_id	doctor_id	patient_name	available_time
101	1	이나라	2025-05-10 14:00
102	1	최지훈	2025-05-11 10:00
103	2	강민준	2025-05-10 09:00
*/


CREATE TABLE doctors (
	id INTEGER, 
	name TEXT, 
	specialty TEXT, 
	email TEXT, 
	PRIMARY KEY (id)
)

/*
3 rows from doctors table:
id	name	specialty	email
1	김재준	내과	kim_int@hospital.com
2	이수진	외과	lee_surg@hospital.com
3	박민수	소아과	park_ped@hospital.com
*/
Invoking: `sql_db_query_checker` with `{'query': "SELECT available_time FROM appointments WHERE doctor_id = (SELECT id FROM doctors WHERE name = '김재준') LIMIT 5"}`


```sql
SELECT available_time

In [ ]:
# 집계 쿼리: 가장 예약이 많은 의사
conversational_agent.invoke(
    {"input": "예약 건수가 가장 많은 의사는 누구야?"},
    config={"configurable": {"session_id": "test_agg"}}
)
# 기대 SQL: SELECT d.name, COUNT(*) AS cnt FROM doctors d
#           JOIN appointments a ON d.id = a.doctor_id
#           GROUP BY d.name ORDER BY cnt DESC LIMIT 1;

# 날짜 필터: 특정 날짜 예약 현황
conversational_agent.invoke(
    {"input": "2025년 5월 10일에 예약된 환자 목록을 알려줘"},
    config={"configurable": {"session_id": "test_date"}}
)
# 기대 SQL: SELECT p.patient_name, d.name AS doctor
#           FROM appointments p JOIN doctors d ON p.doctor_id = d.id
#           WHERE p.available_time LIKE '2025-05-10%';



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


appointments, doctors
Invoking: `sql_db_schema` with `{'table_names': 'appointments'}`



CREATE TABLE appointments (
	app_id INTEGER, 
	doctor_id INTEGER, 
	patient_name TEXT, 
	available_time TEXT, 
	PRIMARY KEY (app_id), 
	FOREIGN KEY(doctor_id) REFERENCES doctors (id)
)

/*
3 rows from appointments table:
app_id	doctor_id	patient_name	available_time
101	1	이나라	2025-05-10 14:00
102	1	최지훈	2025-05-11 10:00
103	2	강민준	2025-05-10 09:00
*/
Invoking: `sql_db_schema` with `{'table_names': 'doctors'}`



CREATE TABLE doctors (
	id INTEGER, 
	name TEXT, 
	specialty TEXT, 
	email TEXT, 
	PRIMARY KEY (id)
)

/*
3 rows from doctors table:
id	name	specialty	email
1	김재준	내과	kim_int@hospital.com
2	이수진	외과	lee_surg@hospital.com
3	박민수	소아과	park_ped@hospital.com
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT d.name, COUNT(a.app_id) as appointment_count \nFROM doctors d \nJOIN appointments a ON d.id = a.do

{'input': '2025년 5월 10일에 예약된 환자 목록을 알려줘',
 'chat_history': [HumanMessage(content='김재준 의사의 이메일이 뭐야?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='김재준 의사의 이메일은 "kim_int@hospital.com"입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='그 의사의 예약 가능 시간은 언제야?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='김재준 의사의 예약 가능 시간은 다음과 같습니다:\n- 2025년 5월 10일 오후 2시\n- 2025년 5월 11일 오전 10시\n- 2025년 5월 13일 오전 11시', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='예약 건수가 가장 많은 의사는 누구야?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='예약 건수가 가장 많은 의사는 김재준이며, 총 3건의 예약이 있습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': '2025년 5월 10일에 예약된 환자 목록은 다음과 같습니다:\n- 이나라\n- 강민준'}